# *Koduppgifter - Kapitel 5*

### **8. Förklara vad nedanstående kod gör**

In [39]:
# NumPy används för att skapa och hantera datasetet.
# PCA importeras från scikit-learn för att kunna minska antalet dimensioner.
import numpy as np
from sklearn.decomposition import PCA

In [40]:
# Skapar ett dataset med 1000 observationer och 3 variabler/features.
# np.random.rand() fyller datasetet med slumpmässiga värden mellan 0 och 1.
X = np.random.rand(1000, 3)

# Skriver ut de första fem raderna för att se hur datasetet ser ut.
print(X[0:5])

[[0.16197452 0.99646295 0.18423439]
 [0.09215317 0.45057925 0.56563162]
 [0.85271242 0.89025816 0.88773628]
 [0.40558727 0.81302236 0.17093416]
 [0.88847381 0.68754785 0.53164057]]


In [41]:
# Skapar en PCA-modell som ska minska antalet dimensioner från 3 till 2.
# n_components=2 betyder att vi behåller två komponenter.
pca = PCA(n_components=2)

# PCA-modellen anpassas till datasetet och omvandlar samtidigt datan från 3 dimensioner till 2 dimensioner.
X2D = pca.fit_transform(X)

# Skriver ut de första fem raderna av den nya 2-dimensionella datan.
print(X2D[0:5])

[[-0.20427356 -0.51277169]
 [-0.30187396 -0.20222053]
 [-0.12494015  0.47502212]
 [ 0.03797075 -0.34848418]
 [ 0.19324686  0.24725357]]


In [42]:
# Omvandlar den 2-dimensionella datan tillbaka till 3 dimensioner.
# Resultatet blir en approximation av den ursprungliga datan eftersom information har förlorats när vi gick från 3 till 2 dimensioner.
X3D_inv = pca.inverse_transform(X2D)

# Kontrollerar om den återskapade datan är tillräckligt lik den ursprungliga datan för att räknas som lika.
# Eftersom PCA tog bort en dimension är resultatet normalt False.
print(np.allclose(X3D_inv, X))

False


#### **Sammanfattning**

Koden visar hur PCA kan användas för att minska antalet dimensioner i ett dataset.
Först skapas ett dataset med `1000` observationer och `3` features med hjälp av `np.random.rand(1000, 3)`. Värdena som skapas är slumpmässiga och ligger mellan 0 och 1. De första fem observationerna skrivs sedan ut för att visa hur datan ser ut.

Därefter skapas en PCA-modell med `n_components=2`. Det betyder att vi vill reducera datan från tre dimensioner till två. Med `fit_transform()` anpassas PCA-modellen efter datan och transformerar samtidigt datasetet till de två nya principalkomponenterna. Resultatet sparas i `X2D`.

Sedan används `inverse_transform()` för att försöka återskapa den ursprungliga datan med tre dimensioner från de två principalkomponenterna. Eftersom vi reducerade från tre dimensioner till två har viss information försvunnit. Den återskapade datan blir därför en approximation av originaldatan och inte exakt likadan.

Till sist används `np.allclose(X3D_inv, X)` för att kontrollera om den återskapade datan är tillräckligt lik originaldatan. Resultatet blir `False`, vilket visar att den återskapade datan inte är tillräckligt lik originaldatan för att räknas som samma.

### **9. PCA på “car_price_dataset.csv”**

In [43]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import root_mean_squared_error

### **Loading Data**

In [44]:
df = pd.read_csv("dataset/car_price_dataset.csv", sep=";")

### **Check the data**

In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Brand         10000 non-null  str    
 1   Model         10000 non-null  str    
 2   Year          10000 non-null  int64  
 3   Engine_Size   10000 non-null  float64
 4   Fuel_Type     10000 non-null  str    
 5   Transmission  10000 non-null  str    
 6   Mileage       10000 non-null  int64  
 7   Doors         10000 non-null  int64  
 8   Owner_Count   10000 non-null  int64  
 9   Price         10000 non-null  int64  
dtypes: float64(1), int64(5), str(4)
memory usage: 1.0 MB


In [46]:
df.describe()

,Year,Engine_Size,Mileage,Doors,Owner_Count,Price
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,2011.543700,3.000560,149239.111800,3.497100,2.991100,8852.96440
std,6.897699,1.149324,86322.348957,1.110097,1.422682,3112.59681
min,2000.000000,1.000000,25.000000,2.000000,1.000000,2000.00000
25%,2006.000000,2.000000,74649.250000,3.000000,2.000000,6646.00000
50%,2012.000000,3.000000,149587.000000,3.000000,3.000000,8858.50000
75%,2017.000000,4.000000,223577.500000,4.000000,4.000000,11086.50000
max,2023.000000,5.000000,299947.000000,5.000000,5.000000,18301.00000


In [47]:
df.head()

,Brand,Model,Year,Engine_Size,Fuel_Type,Transmission,Mileage,Doors,Owner_Count,Price
0,Kia,Rio,2020,4.2,Diesel,Manual,289944,3,5,8501
1,Chevrolet,Malibu,2012,2.0,Hybrid,Automatic,5356,2,3,12092
2,Mercedes,GLA,2020,4.2,Diesel,Automatic,231440,4,2,11171
3,Audi,Q5,2023,2.0,Electric,Manual,160971,2,1,11780
4,Volkswagen,Golf,2003,2.6,Hybrid,Semi-Automatic,286618,3,3,2867


In [48]:
df.isnull().sum()

Brand           0
Model           0
Year            0
Engine_Size     0
Fuel_Type       0
Transmission    0
Mileage         0
Doors           0
Owner_Count     0
Price           0
dtype: int64

In [49]:
df.duplicated().sum()

np.int64(0)

### **Preparing the Data**

In [50]:
X = df.drop("Price", axis=1)
y = df["Price"]

In [51]:
X = pd.get_dummies(X, drop_first=True)

In [52]:
X.head()

,Year,Engine_Size,Mileage,Doors,Owner_Count,Brand_BMW,Brand_Chevrolet,Brand_Ford,Brand_Honda,Brand_Hyundai,...,Model_Sonata,Model_Sportage,Model_Tiguan,Model_Tucson,Model_X5,Fuel_Type_Electric,Fuel_Type_Hybrid,Fuel_Type_Petrol,Transmission_Manual,Transmission_Semi-Automatic
0,2020,4.2,289944,3,5,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
1,2012,2.0,5356,2,3,False,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False
2,2020,4.2,231440,4,2,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2023,2.0,160971,2,1,False,False,False,False,False,...,False,False,False,False,False,True,False,False,True,False
4,2003,2.6,286618,3,3,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,True


### **Train, Validation and Test**

In [53]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=40
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, random_state=40
)

### **Standardization**

In [54]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

### **Models without PCA**

In [55]:
# Linear Regression

linear_regression = LinearRegression()
linear_regression.fit(X_train_scaled, y_train)
linear_pred = linear_regression.predict(X_test_scaled)
linear_rmse = root_mean_squared_error(y_test, linear_pred)

print("Linear Regression RMSE:", linear_rmse)

Linear Regression RMSE: 110.0087654349135


In [56]:
# Decision Tree

decision_tree = DecisionTreeRegressor(random_state=40)
decision_tree.fit(X_train_scaled, y_train)
tree_pred = decision_tree.predict(X_test_scaled)
tree_rmse = root_mean_squared_error(y_test, tree_pred)

print("Decision Tree RMSE:", tree_rmse)

Decision Tree RMSE: 929.3506824659893


In [57]:
# Random Forest

random_forest = RandomForestRegressor(n_estimators=100, random_state=40)
random_forest.fit(X_train_scaled, y_train)
forest_pred = random_forest.predict(X_test_scaled)
forest_rmse = root_mean_squared_error(y_test, forest_pred)

print("Random Forest RMSE:", forest_rmse)

Random Forest RMSE: 590.3526300194656


### **PCA**

In [58]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [59]:
print("Number of components:", pca.n_components_)

Number of components: 35


In [60]:
print("Explained variance:", pca.explained_variance_ratio_.sum())

Explained variance: 0.9591144625837067


### **Models with PCA**

In [61]:
# Linear Regression with PCA

linear_regression_pca = LinearRegression()
linear_regression_pca.fit(X_train_pca, y_train)
linear_pred_pca = linear_regression_pca.predict(X_test_pca)
linear_rmse_pca = root_mean_squared_error(y_test, linear_pred_pca)

print("Linear Regression with PCA RMSE:", linear_rmse_pca)

Linear Regression with PCA RMSE: 869.9647854026765


In [62]:
# Decision Tree with PCA

decision_tree_pca = DecisionTreeRegressor(random_state=40)
decision_tree_pca.fit(X_train_pca, y_train)
tree_pred_pca = decision_tree_pca.predict(X_test_pca)
tree_rmse_pca = root_mean_squared_error(y_test, tree_pred_pca)

print("Decision Tree with PCA RMSE:", tree_rmse_pca)

Decision Tree with PCA RMSE: 2040.7688250999915


In [63]:
# Random Forest with PCA

random_forest_pca = RandomForestRegressor(n_estimators=100, random_state=40)
random_forest_pca.fit(X_train_pca, y_train)
forest_pred_pca = random_forest_pca.predict(X_test_pca)
forest_rmse_pca = root_mean_squared_error(y_test, forest_pred_pca)

print("Random Forest with PCA RMSE:", forest_rmse_pca)

Random Forest with PCA RMSE: 1171.8197135844107


### **Comparing results**

In [64]:
results = pd.DataFrame(
    {
        "Model": ["Linear Regression", "Decision Tree", "Random Forest"],
        "Without PCA": [linear_rmse, tree_rmse, forest_rmse],
        "With PCA": [linear_rmse_pca, tree_rmse_pca, forest_rmse_pca],
    }
)

results

,Model,Without PCA,With PCA
0,Linear Regression,110.008765,869.964785
1,Decision Tree,929.350682,2040.768825
2,Random Forest,590.352630,1171.819714


### **Analys**

PCA reducerade antalet dimensioner till 35 komponenter och behöll cirka 95,91 % av variansen i datan.
För att undersöka hur PCA påverkade resultatet jämfördes samma tre modeller med och utan PCA. Resultatet visar att RMSE ökade för samtliga modeller efter att PCA användes.

`LinearRegression` hade ett RMSE på cirka 110,01 utan PCA och 869,96 med PCA. `DecisionTreeRegressor` ökade från cirka 929,35 till 2040,77, medan `RandomForestRegressor` ökade från cirka 590,35 till 1171,82.
Eftersom ett lägre RMSE innebär mindre prediktionsfel visar resultaten att modellerna presterade sämre efter PCA. PCA förbättrade alltså inte resultatet för någon av de tre modellerna i detta dataset.

En möjlig förklaring är att information som är relevant för att prediktera `Price` försvinner när de ursprungliga variablerna ersätts av ett mindre antal principalkomponenter. Även om PCA behåller cirka 95,91 % av variansen betyder det inte att de resterande cirka 4,09 % är oviktiga för just prediktionen av bilpriset.
Resultatet visar därför att PCA inte automatiskt förbättrar en ML-modell. I det här fallet blev prediktionerna sämre efter dimensionsreduceringen, vilket innebär att de ursprungliga dimensionerna gav bättre resultat för de modeller som testades.